# VisionMaster Hackathon — BDD100K 交通场景目标检测 (YOLO26)

数据：`bdd100k_selected/`，train 7000 / val 2000 / test 1000，图像 1280×720，YOLO 格式标签，9 个类别。

流程：**环境检查 → 生成 data.yaml → (可选)标注可视化 → 训练 → 验证 → test 推理生成 results.csv**

训练和生成 test 结果的 cell 相互独立：推理 cell 会自动寻找最新一次训练的 `best.pt`，可以单独运行。

In [ ]:
# ============ 环境检查 ============
from pathlib import Path
import torch
import ultralytics # type: ignore

ROOT = Path.cwd()
DATA_DIR = ROOT / 'bdd100k_selected'

print('ultralytics:', ultralytics.__version__)
print('torch      :', torch.__version__)
print('cuda       :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert DATA_DIR.exists(), f'数据集目录不存在: {DATA_DIR}'

In [ ]:
# ============ 生成 data.yaml + 数据完整性检查 ============
CLASS_NAMES = ['person', 'rider', 'car', 'truck', 'bus',
               'motorcycle', 'bicycle', 'traffic light', 'traffic sign']

names_block = '\n'.join(f'  {i}: {n}' for i, n in enumerate(CLASS_NAMES))
yaml_text = (
    f'path: {DATA_DIR.as_posix()}\n'
    'train: images/train\n'
    'val: images/val\n'
    'test: images/test\n'
    f'names:\n{names_block}\n'
)
DATA_YAML = ROOT / 'data.yaml'
DATA_YAML.write_text(yaml_text, encoding='utf-8')
print(yaml_text)

for split in ['train', 'val', 'test']:
    n_img = len(list((DATA_DIR / 'images' / split).glob('*.jpg')))
    n_lbl = len(list((DATA_DIR / 'labels' / split).glob('*.txt')))
    print(f'{split:5s}  images={n_img:5d}  labels={n_lbl:5d}')

In [ ]:
# ============ (可选) 随机可视化几张训练标注，确认框和类别对得上 ============
import random
import cv2
import matplotlib.pyplot as plt

samples = random.sample(list((DATA_DIR / 'images' / 'train').glob('*.jpg')), 2)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, img_path in zip(axes, samples):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl_path = DATA_DIR / 'labels' / 'train' / (img_path.stem + '.txt')
    for line in lbl_path.read_text().splitlines():
        c, cx, cy, bw, bh = line.split()
        cx, cy, bw, bh = float(cx) * w, float(cy) * h, float(bw) * w, float(bh) * h
        x1, y1 = int(cx - bw / 2), int(cy - bh / 2)
        x2, y2 = int(cx + bw / 2), int(cy + bh / 2)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, CLASS_NAMES[int(c)], (x1, max(y1 - 4, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
    ax.imshow(img)
    ax.set_title(img_path.name)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 训练

- `yolo26s.pt`：COCO 预训练权重，9 类里 7 类 COCO 本来就有，迁移学习收敛快
- `imgsz=960`：数据集里 traffic light/sign 非常小（原图上仅 13~26px），高分辨率对小目标关键
- `batch=-1`：自动按显存选 batch（本地 6GB 和远程大显存都能自适应）
- 中断后可改 `model = YOLO('runs/yolo26s_960/weights/last.pt')` 并加 `resume=True` 续训

In [ ]:
# ============ 训练 ============
from ultralytics import YOLO # type: ignore

MODEL   = 'yolo26s.pt'
IMGSZ   = 960
EPOCHS  = 80
DEVICE  = 0 if torch.cuda.is_available() else 'cpu'
BATCH   = -1 if torch.cuda.is_available() else 8   # -1 = 自动按显存选择

model = YOLO(MODEL)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    patience=20,          # 20 个 epoch 无提升则早停
    seed=0,
    project='runs',
    name='yolo26s_960',
    exist_ok=True,
)
BEST = Path(results.save_dir) / 'weights' / 'best.pt'
print('best weights:', BEST)

In [ ]:
# ============ 验证集评估（整体 + 每类 mAP + F1 最优阈值） ============
from ultralytics import YOLO

def find_best():
    cands = sorted(ROOT.glob('runs/**/weights/best.pt'), key=lambda p: p.stat().st_mtime)
    assert cands, '未找到训练产物 runs/**/weights/best.pt，请先运行训练 cell'
    return cands[-1]

BEST = globals().get('BEST') or find_best()
best_model = YOLO(str(BEST))
metrics = best_model.val(data=str(DATA_YAML), split='val', imgsz=960,
                         device=0 if torch.cuda.is_available() else 'cpu')

print(f'\nmAP50-95: {metrics.box.map:.4f}   mAP50: {metrics.box.map50:.4f}')
print('-' * 40)
for idx, ap in zip(metrics.box.ap_class_index, metrics.box.maps[metrics.box.ap_class_index]):
    print(f'{CLASS_NAMES[idx]:15s}  mAP50-95 = {ap:.4f}')

# 比赛按 F1 打分：从 F1-Confidence 曲线上找使 mean-F1 最大的置信度阈值
for name, (px, py, *_) in zip(metrics.curves, metrics.curves_results):
    if 'F1' in name:
        f1_mean = py.mean(0)
        BEST_CONF = float(px[f1_mean.argmax()])
        print(f'\n最优置信度阈值: conf={BEST_CONF:.3f}  ->  mean-F1={f1_mean.max():.4f}')

## test 推理 → submission.csv

**Kaggle 提交格式**（来自官方 baseline `test.py`）：每图一行，`pic_name` 带 .jpg 后缀，`results` 为 9 个类别的检测计数用分号连接（class 0~8 顺序），必须覆盖全部 1000 张 test 图：

```
pic_name,results
cabc30fc-fd79926f.jpg,4;0;6;0;0;2;0;0;0
```

> 比赛按 F1 打分且只比计数——框坐标不参与评分，但计数的准确性完全取决于检测质量。
> 置信度阈值用验证 cell 的 `BEST_CONF`；同时保留框级明细 `results.csv` 便于调试分析。

In [ ]:
# ============ test 推理，生成 submission.csv（Kaggle 格式）+ results.csv（框级明细） ============
from collections import Counter
import pandas as pd
from ultralytics import YOLO

def find_best():
    cands = sorted(ROOT.glob('runs/**/weights/best.pt'), key=lambda p: p.stat().st_mtime)
    assert cands, '未找到训练产物 runs/**/weights/best.pt，请先运行训练 cell'
    return cands[-1]

BEST = globals().get('BEST') or find_best()
print('using weights:', BEST)
model = YOLO(str(BEST))

CONF = globals().get('BEST_CONF', 0.25)   # F1 最优阈值，来自验证 cell
print('conf threshold =', CONF)

rows, counts = [], {}
preds = model.predict(
    source=str(DATA_DIR / 'images' / 'test'),
    imgsz=960,
    conf=CONF,
    max_det=300,
    device=0 if torch.cuda.is_available() else 'cpu',
    stream=True,
    verbose=False,
)
for r in preds:
    pic = Path(r.path).name
    counts[pic] = Counter()
    for b in r.boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        counts[pic][int(b.cls)] += 1
        rows.append({
            'image_id': Path(r.path).stem, 'class_id': int(b.cls),
            'confidence': round(float(b.conf), 5),
            'x_min': round(x1, 2), 'y_min': round(y1, 2),
            'x_max': round(x2, 2), 'y_max': round(y2, 2),
        })

pd.DataFrame(rows).to_csv('results.csv', index=False)   # 框级明细（调试用）

test_names = sorted(p.name for p in (DATA_DIR / 'images' / 'test').glob('*.jpg'))
sub = pd.DataFrame([
    {'pic_name': n,
     'results': ';'.join(str(counts.get(n, Counter()).get(i, 0)) for i in range(9))}
    for n in test_names])
sub.to_csv('submission.csv', index=False)
print(f'submission.csv: {len(sub)} 行 (需=1000)   results.csv: {len(rows)} 框')
sub.head(5)